<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-09-ai-agents/lesson-9.4-memory-systems/notebooks/exercises-9_4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 9.4: Memory Systems for Agents

*Module 9 · AI Agents & Autonomous Systems*

> An agent without memory forgets everything between turns. With memory, it remembers your name, your preferences, what you discussed last week, and lessons learned from past mistakes. This lesson builds four memory systems from scratch: conversation buffers, summarization, vector-backed long-term memory, and episodic memory — each solving a different problem.

`Buffer Memory` · `Summarization` · `Vector Memory` · `Episodic` · `60 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-9.4.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Buffer Memory — The Simplest Approach — `01_buffer_memory.py`
2. Step 2 — Summarization Memory — Compress, Don’t Drop — `02_summary_memory.py`
3. Step 3 — Vector Memory — Semantic Long-Term Recall — `03_vector_memory.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai numpy


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Buffer Memory — The Simplest Approach

Keep the last N messages. Drop everything else. Fast but forgetful.

**`01_buffer_memory.py`** · _Block 1 of 3_

BUFFER MEMORY — Keep last N messages


In [ ]:
# BUFFER MEMORY — Keep last N messages
import google.generativeai as genai, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class BufferMemory:
    """Sliding window: keep last N message pairs."""
    def __init__(self, window=6):
        self.messages = []
        self.window = window
        self.model = genai.GenerativeModel("gemini-2.0-flash")

    def chat(self, user_msg):
        self.messages.append({"role":"user", "text":user_msg})
        # Build context from window
        ctx = "\n".join(f"{m['role']}: {m['text']}" for m in self.messages[-self.window:])
        resp = self.model.generate_content(f"Conversation:\n{ctx}\nassistant:").text.strip()
        self.messages.append({"role":"assistant", "text":resp})
        # Trim
        if len(self.messages) > self.window * 2:
            self.messages = self.messages[-self.window:]
        return resp

    def stats(self):
        return {"stored":len(self.messages), "window":self.window}

bot = BufferMemory(window=4)
print("Buffer Memory (window=4):\n")
for msg in ["My name is Priya", "I'm interested in GenAI",
            "What courses do you have?", "How much do they cost?",
            "What's my name?"]:  # may forget if window too small!
    resp = bot.chat(msg)
    print(f"  You: {msg}")
    print(f"  Bot: {resp[:80]}  [{bot.stats()}]\n")


> **What just happened?** With window=4, the agent keeps only the last 4 messages. “What’s my name?” at turn 5 might fail because “My name is Priya” was dropped. Buffer memory is fast and simple but FORGETS. It’s fine for 3-5 turn conversations. For longer conversations, you need summarization.


### Step 2 · Summarization Memory — Compress, Don’t Drop

When the buffer overflows, summarize old messages instead of deleting them.

**`02_summary_memory.py`** · _Block 2 of 3_

SUMMARIZATION MEMORY — Compress old context


In [ ]:
# SUMMARIZATION MEMORY — Compress old context
import google.generativeai as genai, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class SummaryMemory:
    """Summarize old messages, keep recent ones fresh."""
    def __init__(self, recent_window=4):
        self.messages = []
        self.summary = ""
        self.window = recent_window
        self.model = genai.GenerativeModel("gemini-2.0-flash")

    def _summarize(self, old_msgs):
        text = "\n".join(f"{m['role']}: {m['text'][:80]}" for m in old_msgs)
        prev = f"Previous summary: {self.summary}\n\n" if self.summary else ""
        resp = self.model.generate_content(
            f"{prev}Summarize this conversation in 2-3 sentences. Keep names, numbers, preferences:\n{text}")
        self.summary = resp.text.strip()

    def chat(self, user_msg):
        self.messages.append({"role":"user", "text":user_msg})
        # Compress if overflow
        if len(self.messages) > self.window * 2:
            old = self.messages[:-self.window]
            self._summarize(old)
            self.messages = self.messages[-self.window:]
        # Build context
        recent = "\n".join(f"{m['role']}: {m['text']}" for m in self.messages[-self.window:])
        ctx = f"Summary of earlier conversation: {self.summary}\n\nRecent:\n{recent}" if self.summary else recent
        resp = self.model.generate_content(f"{ctx}\nassistant:").text.strip()
        self.messages.append({"role":"assistant", "text":resp})
        return resp

bot = SummaryMemory(recent_window=4)
print("Summarization Memory:\n")
for msg in ["My name is Priya and I'm from Hyderabad", "I want to learn GenAI",
            "My budget is 15000 rupees", "I know Python already",
            "What course do you recommend?", "Can I pay in EMI?",
            "What's my name and budget?"]:  # summary retains this!
    resp = bot.chat(msg)
    print(f"  You: {msg}")
    print(f"  Bot: {resp[:80]}\n")
print(f"  Summary: {bot.summary[:120]}")


### Step 3 · Vector Memory — Semantic Long-Term Recall

Embed facts, store as vectors, retrieve by similarity. Remembers across sessions.

**`03_vector_memory.py`** · _Block 3 of 3_

VECTOR MEMORY — Semantic long-term recall


In [ ]:
# VECTOR MEMORY — Semantic long-term recall
import google.generativeai as genai, numpy as np, os, json
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class VectorMemory:
    """Store facts as embeddings. Retrieve by semantic similarity."""
    def __init__(self):
        self.facts = []  # {"text":..., "embedding":..., "timestamp":...}
        self.model = genai.GenerativeModel("gemini-2.0-flash")

    def _embed(self, text):
        return np.array(genai.embed_content(model="models/text-embedding-004", content=text)["embedding"])

    def store(self, fact):
        """Store a fact with its embedding."""
        from datetime import datetime
        self.facts.append({"text":fact, "embedding":self._embed(fact),
                          "time":datetime.now().isoformat()})

    def recall(self, query, k=3):
        """Retrieve most relevant facts by cosine similarity."""
        if not self.facts: return []
        qe = self._embed(query)
        scores = [(f, np.dot(qe,f["embedding"])/(np.linalg.norm(qe)*np.linalg.norm(f["embedding"])))
                  for f in self.facts]
        scores.sort(key=lambda x:-x[1])
        return [s[0]["text"] for s in scores[:k]]

    def extract_and_store(self, conversation_turn):
        """LLM extracts storable facts from a conversation turn."""
        resp = self.model.generate_content(
            f"Extract key facts from this message. Return JSON array of strings.\n"
            f"Only factual info worth remembering (names, preferences, numbers).\n"
            f"Message: {conversation_turn}\nFacts:")
        raw = resp.text.strip()
        if raw.startswith("```"): raw = raw.split("\n",1)[1].rsplit("```",1)[0]
        try:
            facts = json.loads(raw)
            for f in facts: self.store(f)
            return facts
        except: return []

# Demo
mem = VectorMemory()
print("Vector Memory:\n")

# Store facts from a conversation
for msg in ["My name is Priya, I'm a Python developer in Hyderabad",
            "I'm interested in GenAI and my budget is 15000",
            "I prefer evening classes after 7 PM"]:
    facts = mem.extract_and_store(msg)
    print(f"  Stored from: '{msg[:40]}...' → {facts}")

# Recall by semantic query (days/sessions later!)
print(f"\n  Recall:")
for q in ["What's her budget?", "When does she prefer classes?", "What does she know?"]:
    recalled = mem.recall(q, k=2)
    print(f"    '{q}' → {recalled}")


> **What just happened?** The LLM extracted facts (“Priya”, “Python developer”, “budget 15000”, “evening classes”) and embedded them. Days later, “What’s her budget?” retrieves “budget is 15000” by cosine similarity. Vector memory = cross-session, semantic recall. The agent remembers users across conversations. This is how Claude, ChatGPT, and Gemini implement “memory.”


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
